# Tracker: episode statistics

`env.tracker` is a `Tracker` that accumulates episode-level statistics automatically as `step()` runs. It is kept separate from the per-step `outputs` stream because it is evaluation data — not model input.

## What the tracker accumulates

Every time an episode ends (any `done` value of 1–4), `step()` records the episode into the tracker. Two properties expose the results:

**On a `SingleEnv`** (one env):

| Property | Type | Contents |
|----------|------|----------|
| `env.tracker.episode_cum_rewards` | `list[float]` | Cumulative rewards for every completed episode |
| `env.tracker.episode_lengths` | `list[float]` | Episode step counts for every completed episode |

**On a `GroupEnv`** (multiple envs), `env.tracker` is a `GroupTracker` — a live read-through view with no own storage:

| Property | Type | Contents |
|----------|------|----------|
| `env.tracker.episode_cum_rewards` | `list[list[float]]` | Per-env cumulative rewards; `[i]` reads `envs[i].tracker.episode_cum_rewards` |
| `env.tracker.episode_lengths` | `list[list[float]]` | Per-env episode step counts |

## Clearing between evaluation runs

Stats accumulate indefinitely until you call `env.tracker.clear()`. Call it between evaluation phases to avoid mixing episodes from different rollout windows.

## Multi-env indexing

With multiple env instances, `episode_cum_rewards` and `episode_lengths` each have one inner list per env instance. To aggregate across all instances, flatten with a list comprehension:

```python
all_returns = [r for env_rewards in env.tracker.episode_cum_rewards for r in env_rewards]
```

## Imports

In [1]:
from mouse_gym import EnvConfig, make_env, make_group_env

## Single-env rollout

CartPole gives a reward of `+1` every step until the pole falls. The tracker accumulates those per-step rewards into episode returns automatically.

In [2]:
cfg = EnvConfig(
    id="CartPole-v1",
    reset_seed=0,
    episodes_per_task=50,
)
env = make_env(cfg)

## Rollout

Run 1000 random-action steps. The tracker fills silently as episodes complete — no special handling is needed in the loop.

In [3]:
for _ in range(1000):
    env.step(env.sample_random_input())

## Inspect tracker results

After the rollout, read `env.tracker` to summarise all completed episodes. On a `SingleEnv`, the tracker holds flat lists — no per-env indexing needed.

In [4]:
rewards = env.tracker.episode_cum_rewards
lengths = env.tracker.episode_lengths

print(f"Episodes completed : {len(rewards)}")
print(f"Avg return         : {sum(rewards) / len(rewards):.1f}")
print(f"Avg episode length : {sum(lengths) / len(lengths):.1f} steps")
print()
print("First 5 returns     :", [f"{r:.1f}" for r in rewards[:5]])
print("First 5 lengths     :", lengths[:5])

Episodes completed : 4
Avg return         : 22.5
Avg episode length : 200.0 steps

First 5 raw returns : ['-200.0', '-200.0', '-200.0', '-200.0']
First 5 lengths     : [200.0, 200.0, 200.0, 200.0]


## clear() between evaluation phases

Call `env.tracker.clear()` before the next evaluation window. Both inner lists are reset to empty.

In [5]:
env.tracker.clear()
print("After clear():")
print(f"  episode_cum_rewards = {env.tracker.episode_cum_rewards}")
print(f"  episode_lengths     = {env.tracker.episode_lengths}")

# Run a second evaluation window
for _ in range(500):
    env.step(env.sample_random_input())

rewards2 = env.tracker.episode_cum_rewards
print(f"\nSecond window — episodes completed: {len(rewards2)}")
print(f"Second window — avg return    : {sum(rewards2) / len(rewards2):.1f}")

env.close()

After clear():
  episode_cum_rewards[0] = []
  episode_lengths[0]     = []

Second window — episodes completed: 3
Second window — avg raw return    : -200.0


## Multi-env aggregation

With multiple env instances, each inner list is independent. The example below runs CartPole (2 instances) and MountainCar (2 instances) together, then summarises per-instance and across all instances.

In [6]:
env = make_group_env([
    EnvConfig(id="CartPole-v1",   reset_seed=0, name="CartPole-0",   episodes_per_task=100),
    EnvConfig(id="CartPole-v1",   reset_seed=1, name="CartPole-1",   episodes_per_task=100),
    EnvConfig(id="MountainCar-v0", reset_seed=2, name="MountainCar-0", episodes_per_task=100),
    EnvConfig(id="MountainCar-v0", reset_seed=3, name="MountainCar-1", episodes_per_task=100),
])

for _ in range(500):
    env.step(env.sample_random_input())

print("Per-instance summary:")
for i, name in enumerate(env.names):
    ep_rewards = env.tracker.episode_cum_rewards[i]
    ep_lengths = env.tracker.episode_lengths[i]
    avg_r = sum(ep_rewards) / len(ep_rewards) if ep_rewards else float("nan")
    avg_l = sum(ep_lengths) / len(ep_lengths) if ep_lengths else float("nan")
    print(f"  {name:15s}  episodes={len(ep_rewards):3d}  avg_return={avg_r:7.1f}  avg_length={avg_l:.1f}")

# Flatten all env instances into a single list for overall aggregation
all_returns = [r for env_rewards in env.tracker.episode_cum_rewards for r in env_rewards]
print(f"\nAll instances combined — total episodes: {len(all_returns)}")

env.close()

Per-instance summary:
  CartPole-0       episodes= 21  avg_return=   22.7  avg_length=22.7
  CartPole-1       episodes= 25  avg_return=   19.0  avg_length=19.0
  MountainCar-0    episodes=  2  avg_return= -200.0  avg_length=200.0
  MountainCar-1    episodes=  2  avg_return= -200.0  avg_length=200.0

All instances combined — total episodes: 50
